In [86]:
#install dependencies, you can uncomment if needed
# %pip install scikit-image
# %pip install matplotlib
# %pip install numpy

# print("\nFinished installing scikit, matplotlib, and numpy\n")

import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import numpy as np
from skimage.transform import resize
import random
import os
from PIL import Image
import pandas as pd

## Preprocessing: Median filter and Gaussian Filter

In [87]:
import math
import numpy as np

#function for convolution, based on the matlab code showed in class
#This is using zero padding, so the result image is slightly larger than the original image.

def twoD_convolution(image, kernel):
    image_height = image.shape[0]
    image_width = image.shape[1]

    kernel_height = kernel.shape[0]
    kernel_width = kernel.shape[1]
    result_height = image_height + kernel_height - 1
    result_width = image_width + kernel_width - 1
    
    # Initialize result with zeros
    result = np.zeros((result_height, result_width))
    for i in range(image_height):
        for j in range(image_width):
            # For each pixel in the image, multiply it by the entire kernel
            # and add it to the corresponding neighborhood in the result
            result[i : i + kernel_height, j : j + kernel_width] += image[i, j] * kernel

    pad_h = kernel_height // 2
    pad_w = kernel_width // 2
    return result[pad_h : pad_h + image_height, pad_w : pad_w + image_width]

def median_filter(img, kernel_size):
    pad = kernel_size // 2
    #pad the image with zeros on the edges.
    padded = np.pad(img, pad, mode='constant', constant_values=0)

    #initialize the result matrix with zeros (same size as original image)
    result = np.zeros_like(img)
    
    #replace each pixel with the median of the neighboring pixels (inside the kernel)
    #this loops through the entire image
    for i in range(img.shape[0]):
        for j in range(img.shape[1]):
            region = padded[i:i+kernel_size, j:j+kernel_size]
            result[i, j] = np.median(region)
    
    return result

def gaussian_filter_equation(x,y,sigma):
    return math.exp(-(x**2 + y**2) / (2 * sigma**2))

def gaussian_filter(img, kernel_size, sigma):
    k = kernel_size // 2
    kernel = np.zeros((kernel_size, kernel_size))
    for i in range(kernel_size):
        for j in range(kernel_size):
            kernel[i, j] = gaussian_filter_equation(i - k, j - k, sigma)
    kernel /= kernel.sum()
    return twoD_convolution(img, kernel)


### Reading in the images, converting them to grayscale using the Luminosity method, and using the filters.

In [88]:
from matplotlib.image import imread
from PIL import Image
def luminosity(image):
    r = 0.21
    g = 0.72
    b = 0.07
    img = image.copy()
    #create 3 copies of the original image.
    red = img[:, :, 0]
    green = img[:, :, 1]
    blue = img[:, :, 2]

    return (red* r) + (green * g) + (blue * b)

if not os.path.exists("median_filtered_images"):
    os.makedirs("median_filtered_images")
    print(f"Created folder: median_filtered_images")
else:
    print(f"Folder median_filtered_images already exists.")

# There are 10 dataset images, all named the same way (img_1, img_2, img_3...)
print("Median Filtering 10 Images\n")
for i in range(1, 11):
    print("Filtering Image " + str(i))
    #luminsoity -> median filter -> save the result to median_filtered_images folder
    current_image = luminosity(imread("dataset_images/img_" + str(i) + ".jpg"))
    median_filtered = median_filter(current_image, 5) #kernel size of 5
    res_img = Image.fromarray(median_filtered.astype(np.uint8))
    res_img.convert("L").save(f"median_filtered_images/median_img_{i}.jpg")
    # plt.imsave("median_filtered_images/median_img_" + str(i) + ".jpg", median_filtered, cmap='gray')




Created folder: median_filtered_images
Median Filtering 10 Images

Filtering Image 1
Filtering Image 2
Filtering Image 3
Filtering Image 4
Filtering Image 5
Filtering Image 6
Filtering Image 7
Filtering Image 8
Filtering Image 9
Filtering Image 10


In [89]:
if not os.path.exists("median_gaussian_filtered_images"):
    os.makedirs("median_gaussian_filtered_images")
    print(f"Created folder: median_gaussian_filtered_images")
else:
    print(f"Folder median_gaussian_filtered_images already exists.")
    
print("Gaussian Filtering 10 Images\n")
for i in range(1, 11):
    print("Filtering Image " + str(i))
    #gaussian filter -> save the result to median_gaussian_filtered_images folder
    current_image = imread("median_filtered_images/median_img_" + str(i) + ".jpg")
    gaussian_filtered = gaussian_filter(current_image, 5, 1) #kernel size of 5 and sigma of 1
    res_img = Image.fromarray(gaussian_filtered.astype(np.uint8))
    res_img.convert("L").save(f"median_gaussian_filtered_images/median_gaussian_img_{i}.jpg")

Created folder: median_gaussian_filtered_images
Gaussian Filtering 10 Images

Filtering Image 1
Filtering Image 2
Filtering Image 3
Filtering Image 4
Filtering Image 5
Filtering Image 6
Filtering Image 7
Filtering Image 8
Filtering Image 9
Filtering Image 10


### Define the piecewise contrast stretching function:

In [90]:
def piecewise_contrast(img, t):
    image = img.copy()
    image = image.astype(np.float64)
    L_min = 0
    L_max = 255
    i_min = np.min(image)
    i_max = np.max(image)

    result = np.zeros_like(image)
    less_than_t = image <= t
    higher_than_t = image > t
    
    result[less_than_t] = ((image[less_than_t] - i_min) * (t - L_min) / (t - i_min + 0.1)) + L_min
    result[higher_than_t] = ((image[higher_than_t] - t + 1) *(L_max - t+1) / (i_max - t+1)) + t + 1
    return result

### Part A: EME (Contrast Ratio Only)
$$EME = \frac{1}{k_1 k_2} \sum_{k=1}^{k_1} \sum_{l=1}^{k_2} \left( \frac{I_{max,k,l}}{I_{min,k,l} + c} \right)$$

In [91]:
# takes in image, M and N (aka k1 and k2).
#M = vertical blocks, N = horizontal
def get_eme(img, vertical_blocks, horizontal_blocks):
    rows, cols = img.shape
    image = (img.copy()).astype(float)  #I don't want this function to modify the image passed in, so I make a copy and convert that to float.
   
    # floor division to get the number of rows and columns in each block.
    block_rows = rows//vertical_blocks
    block_cols = cols//horizontal_blocks

    eme = 0.0
    for r in range(vertical_blocks):
        for c in range(horizontal_blocks):
           row_start = r*block_rows
           row_end = row_start + block_rows + 1
           col_start = c*block_cols
           col_end = col_start + block_cols + 1

           block = image[row_start:row_end, col_start:col_end]  #Extract the current block of the image.
           block_max_intensity = np.max(block)  
           block_min_intensity = np.min(block) 

           block_min_intensity += 0.1   #(the denomimator, c= 0.1 so we add it) 

           eme += (block_max_intensity / block_min_intensity)  #Apply the EME formula and accumulate the result.    
            
    return float(eme) / (vertical_blocks * horizontal_blocks)  #Average the EME value by dividing by the total number of blocks. (1/MN)

part_a_results = []  #will contain all t values and their EME values, including optimal t. For every image. 
def eme_for_all_images(results_list, img_folder):
    for i in range(0, 10):
        print("Processing image " + str(i+1))
        current_image_max_eme = -1
        current_image_t_optimal = -1
        t_eme_pairs = {}
        if(img_folder == "median_gaussian_filtered_images"):
            img_name = f"median_gaussian_filtered_images/median_gaussian_img_{i+1}.jpg"
        elif(img_folder == "median_filtered_images"):
            img_name = f"median_filtered_images/median_img_{i+1}.jpg"
        elif(img_folder == "dataset_images"):
            img_name = f"dataset_images/img_{i+1}.jpg"

        for t in range(0, 256): #loop through all possible t values (0-255)
            if(img_folder == "dataset_images"):
                current_image = luminosity(imread(img_name))
            else:
                current_image = imread(img_name)
            if(np.max(current_image) <= 1.0):
                current_image = current_image * 255.0
            contrast_stretched_image = piecewise_contrast(current_image, t)
            eme = get_eme(contrast_stretched_image, 8, 8) #use 8x8 blocks
            
            if eme > current_image_max_eme:
                current_image_max_eme = eme
                current_image_t_optimal = t

            t_eme_pairs[t] = eme
        results_list.append({
                "image_#": i,
                "t_optimal": current_image_t_optimal,
                "max_eme": current_image_max_eme,
                "all_values": t_eme_pairs  # for later plotting of t and eme
            })   
    print("\n")
    return results_list 

#to show what the results look like.
#It's pretty long so I won't print the results for every image
part_a_results = eme_for_all_images(part_a_results, "median_gaussian_filtered_images")
print(part_a_results[0])

Processing image 1
Processing image 2
Processing image 3
Processing image 4
Processing image 5
Processing image 6
Processing image 7
Processing image 8
Processing image 9
Processing image 10


{'image_#': 0, 't_optimal': 0, 'max_eme': 439.8376662521662, 'all_values': {0: 439.8376662521662, 1: 439.76857196322567, 2: 439.69893144765985, 3: 439.6287382751637, 4: 439.55798591604116, 5: 439.48666773934394, 6: 439.52815538719244, 7: 439.46955131892844, 8: 439.3944641357852, 9: 439.31896075700354, 10: 439.27908854939676, 11: 439.2020389461555, 12: 439.1560475961921, 13: 439.1015559195359, 14: 439.03810828558585, 15: 438.98522893328624, 16: 438.94348359180606, 17: 438.88518987051526, 18: 438.8208137698231, 19: 438.73643270177166, 20: 438.6625887973133, 21: 438.5767066095119, 22: 438.1579590579783, 23: 437.74326007357087, 24: 437.67327709855266, 25: 437.60479900236, 26: 437.19837621417474, 27: 437.1219783751649, 28: 437.0443166253493, 29: 436.63404615953783, 30: 436.56006146460595, 31: 436.4898

### Part C: EMEE
$$EMEE = \frac{1}{k_1 k_2} \sum_{k=1}^{k_1} \sum_{l=1}^{k_2} \alpha \ln \left( \frac{I_{max,k,l}}{I_{min,k,l} + c} \right)$$

In [92]:
# takes in image, M and N (aka k1 and k2).
#M = vertical blocks, N = horizontal
def get_emee(img, vertical_blocks, horizontal_blocks, alpha):
    rows, cols = img.shape
    image = (img.copy()).astype(float)  #I don't want this function to modify the image passed in, so I make a copy and convert that to float.
   
    # floor division to get the number of rows and columns in each block.
    block_rows = rows//vertical_blocks
    block_cols = cols//horizontal_blocks

    emee = 0.0
    for r in range(vertical_blocks):
        for c in range(horizontal_blocks):
           row_start = r*block_rows
           row_end = row_start + block_rows + 1
           col_start = c*block_cols
           col_end = col_start + block_cols + 1

           block = image[row_start:row_end, col_start:col_end]  #Extract the current block of the image.
           block_max_intensity = np.max(block) + 0.1
           block_min_intensity = np.min(block) + 0.1

           block_min_intensity += 0.1   #(the denomimator, c= 0.1 so we add it) 

           emee += alpha * np.log(block_max_intensity / block_min_intensity)  #Apply the EME formula and accumulate the result.    
            
    return float(emee) / (vertical_blocks * horizontal_blocks)  #Average the EME value by dividing by the total number of blocks. (1/MN)

part_c_results = []  #will contain all t values and their EME values, including optimal t. For every image.
def emee_for_all_images(results_list, img_folder): 
    for i in range(0,10):
        print("Processing image " + str(i+1))
        current_image_max_eme = -1
        current_image_t_optimal = -1
        t_eme_pairs = {}
        if(img_folder == "median_gaussian_filtered_images"):
            img_name = f"median_gaussian_filtered_images/median_gaussian_img_{i+1}.jpg"
        elif(img_folder == "median_filtered_images"):
            img_name = f"median_filtered_images/median_img_{i+1}.jpg"
        elif(img_folder == "dataset_images"):
            img_name = f"dataset_images/img_{i+1}.jpg"

        for t in range(0, 256): #loop through all possible t values (0-255)
            if(img_folder == "dataset_images"):
                current_image = luminosity(imread(img_name))
            else:
                current_image = imread(img_name)
            if(np.max(current_image) <= 1.0):
                current_image = current_image * 255.0
            contrast_stretched_image = piecewise_contrast(current_image, t)
            eme = get_emee(contrast_stretched_image, 8, 8, 0.5) #use 8x8 blocks and alpha of 0.5
            
            if eme > current_image_max_eme:
                current_image_max_eme = eme
                current_image_t_optimal = t

            t_eme_pairs[t] = eme
        results_list.append({
                "image_#": i,
                "t_optimal": current_image_t_optimal,
                "max_eme": current_image_max_eme,
                "all_values": t_eme_pairs  # for later plotting of t and eme
            })
    print("\n") 
    return results_list   

#to show what the results look like.
#It's pretty long so I won't print the results for every image
part_c_results = emee_for_all_images(part_c_results, "median_gaussian_filtered_images")
print(part_c_results[0])

Processing image 1
Processing image 2
Processing image 3
Processing image 4
Processing image 5
Processing image 6
Processing image 7
Processing image 8
Processing image 9
Processing image 10


{'image_#': 0, 't_optimal': 25, 'max_eme': 1.7446802467712141, 'all_values': {0: 1.7209232564073496, 1: 1.721084819393706, 2: 1.7212480555992364, 3: 1.7214129914254934, 4: 1.7215796538391794, 5: 1.7217480703875585, 6: 1.7265834408906633, 7: 1.7287169051877822, 8: 1.7287824884461003, 9: 1.728858014577913, 10: 1.730431874405112, 11: 1.7304974883624196, 12: 1.7330958463249946, 13: 1.7354822630134923, 14: 1.7365874098110088, 15: 1.7386421514085533, 16: 1.740551714858488, 17: 1.7423310937004293, 18: 1.743992869256481, 19: 1.7438991136371018, 20: 1.7445939807669513, 21: 1.7444952584708824, 22: 1.7436782834057172, 23: 1.742903287290104, 24: 1.7434906977717897, 25: 1.7446802467712141, 26: 1.7445804501703104, 27: 1.7444839010235405, 28: 1.7443861558793812, 29: 1.7437380924544794, 30: 1.7436511117342852, 3

### Part E: AME
$$AME = \frac{1}{k_1 k_2} \sum_{k=1}^{k_1} \sum_{l=1}^{k_2} \ln \left( \frac{I_{max,k,l} - I_{min,k,l}}{I_{max,k,l} + I_{min,k,l} + c} \right)$$

In [93]:
# takes in image, M and N (aka k1 and k2).
#M = vertical blocks, N = horizontal
def get_ame(img, vertical_blocks, horizontal_blocks):
    rows, cols = img.shape
    image = (img.copy()).astype(float)  #I don't want this function to modify the image passed in, so I make a copy and convert that to float.
   
    # floor division to get the number of rows and columns in each block.
    block_rows = rows//vertical_blocks
    block_cols = cols//horizontal_blocks
    ame = 0.0
    for r in range(vertical_blocks):
        for c in range(horizontal_blocks):

           row_start = r*block_rows
           row_end = row_start + block_rows + 1
           col_start = c*block_cols
           col_end = col_start + block_cols + 1

           block = image[row_start:row_end, col_start:col_end]  #Extract the current block of the image.
           block_max_intensity = np.max(block) 
           block_min_intensity = np.min(block)
           numerator = block_max_intensity - block_min_intensity
           denominator = block_max_intensity + block_min_intensity + 0.1
           block_min_intensity += 0.1   #(the denomimator, c= 0.1 so we add it)

           if(numerator/denominator) <= 0: #to avoid log of 0 (undefined)
               continue
           else:
            ame +=  np.log(numerator/denominator)  #Apply the EME formula and accumulate the result.    
            
    return float(ame) / (vertical_blocks * horizontal_blocks)  #Average the EME value by dividing by the total number of blocks. (1/MN)

part_e_results = []  #will contain all t values and their EME values, including optimal t. For every image. 
def ame_for_all_images(results_list, img_folder):
    for i in range(0, 10):
        print("Processing image " + str(i+1))
        current_image_max_ame = -1
        current_image_t_optimal = -1
        t_ame_pairs = {}
        if(img_folder == "median_gaussian_filtered_images"):
            img_name = f"median_gaussian_filtered_images/median_gaussian_img_{i+1}.jpg"
        elif(img_folder == "median_filtered_images"):
            img_name = f"median_filtered_images/median_img_{i+1}.jpg"
        elif(img_folder == "dataset_images"):
            img_name = f"dataset_images/img_{i+1}.jpg"

        for t in range(0, 256): #loop through all possible t values (0-255)
            if(img_folder == "dataset_images"):
                current_image = luminosity(imread(img_name))
            else:
                current_image = imread(img_name)

            if(np.max(current_image) <= 1.0):
                current_image = current_image * 255.0
            contrast_stretched_image = piecewise_contrast(current_image, t)
            ame = get_ame(contrast_stretched_image, 8, 8) #use 8x8 blocks and alpha of 0.5
            
            if ame > current_image_max_ame:
                current_image_max_ame = ame
                current_image_t_optimal = t

            t_ame_pairs[t] = ame
        results_list.append({
                "image_#": i,
                "t_optimal": current_image_t_optimal,
                "max_eme": current_image_max_ame,
                "all_values": t_ame_pairs  # for later plotting of t and ame
            })
    print("\n") 
    return results_list   

#to show what the results look like.
#It's pretty long so I won't print the results for every image
part_e_results = ame_for_all_images(part_e_results, "median_gaussian_filtered_images")
print(part_e_results[0])

Processing image 1
Processing image 2
Processing image 3
Processing image 4
Processing image 5
Processing image 6
Processing image 7
Processing image 8
Processing image 9
Processing image 10


{'image_#': 0, 't_optimal': 36, 'max_eme': -0.17220497205600066, 'all_values': {0: -0.19404001222027595, 1: -0.193853482983824, 2: -0.19366537897134906, 3: -0.19347568008490604, 4: -0.1932843658817716, 5: -0.19309141556698925, 6: -0.19069492025513488, 7: -0.18820771803656108, 8: -0.18808426109860168, 9: -0.1879543321939741, 10: -0.18731308569395538, 11: -0.18718229869985192, 12: -0.18525156837656828, 13: -0.1822174061390946, 14: -0.18152807527044718, 15: -0.17987263499909922, 16: -0.17905098277237722, 17: -0.17785286363320854, 18: -0.176408342129761, 19: -0.17640243161123925, 20: -0.17593994561978685, 21: -0.17593552667779794, 22: -0.17593528035222528, 23: -0.17593256995200438, 24: -0.1754982218074777, 25: -0.17390798570128002, 26: -0.17341459805492085, 27: -0.17343598881208847, 28: -0.1733766636

### Saving the results for each image. (Image_#, Optimal t for each EME method, and EME value at optimal t)

In [94]:
import pandas as pd


#So far we calculated eme for the fully preprocessed images (median AND gaussian filtered)
#This calculates EMEs for the orignal images with luminosity method and after median filtering

print("EME for original grayscale images\n")
part_a_results_original_grayscale = eme_for_all_images([], "dataset_images")
part_c_results_original_grayscale = emee_for_all_images([], "dataset_images")
part_e_results_original_grayscale = ame_for_all_images([], "dataset_images")

print("EME for median filtered images\n")
part_a_results_median_filtered = eme_for_all_images([], "median_filtered_images")
part_c_results_median_filtered = emee_for_all_images([], "median_filtered_images")
part_e_results_median_filtered = ame_for_all_images([], "median_filtered_images")

rows = []
for i in range(10):
    row = {
        "image_#": i + 1,
        #results for median gaussian filtered images
        "t_optimal_median_gaussian_eme": part_a_results[i]["t_optimal"],
        "eme_at_t0_median_gaussian": part_a_results[i]["all_values"][0],
        "eme_at_t_optimal_median_gaussian": part_a_results[i]["max_eme"],
        "t_optimal_emee_median_gaussian": part_c_results[i]["t_optimal"],
        "emee_at_t0_median_gaussian": part_c_results[i]["all_values"][0],
        "emee_at_t_optimal_median_gaussian": part_c_results[i]["max_eme"],
        "t_optimal_ame_median_gaussian": part_e_results[i]["t_optimal"],
        "ame_at_t0_median_gaussian": part_e_results[i]["all_values"][0],
        "ame_at_t_optimal_median_gaussian": part_e_results[i]["max_eme"],

        #store results for original grayscale images
        "t_optimal_eme_original_grayscale": part_a_results_original_grayscale[i]["t_optimal"],
        "eme_at_t0_original_grayscale": part_a_results_original_grayscale[i]["all_values"][0],
        "eme_at_t_optimal_original_grayscale": part_a_results_original_grayscale[i]["max_eme"],
        "t_optimal_emee_original_grayscale": part_c_results_original_grayscale[i]["t_optimal"],
        "emee_at_t0_original_grayscale": part_c_results_original_grayscale[i]["all_values"][0],
        "emee_at_t_optimal_original_grayscale": part_c_results_original_grayscale[i]["max_eme"],
        "t_optimal_ame_original_grayscale": part_e_results_original_grayscale[i]["t_optimal"],
        "ame_at_t0_original_grayscale": part_e_results_original_grayscale[i]["all_values"][0],
        "ame_at_t_optimal_original_grayscale": part_e_results_original_grayscale[i]["max_eme"],

        #results for median filtered images
        "t_optimal_eme_median_filtered": part_a_results_median_filtered[i]["t_optimal"],
        "eme_at_t0_median_filtered": part_a_results_median_filtered[i]["all_values"][0],
        "eme_at_t_optimal_median_filtered": part_a_results_median_filtered[i]["max_eme"],
        "t_optimal_emee_median_filtered": part_c_results_median_filtered[i]["t_optimal"],
        "emee_at_t0_median_filtered": part_c_results_median_filtered[i]["all_values"][0],
        "emee_at_t_optimal_median_filtered": part_c_results_median_filtered[i]["max_eme"],
        "t_optimal_ame_median_filtered": part_e_results_median_filtered[i]["t_optimal"],
        "ame_at_t0_median_filtered": part_e_results_median_filtered[i]["all_values"][0],
        "ame_at_t_optimal_median_filtered": part_e_results_median_filtered[i]["max_eme"],
    }
    rows.append(row)

df = pd.DataFrame(rows)
df.to_csv("results_summary.csv", index=False)
print(df)

EME for original grayscale images

Processing image 1
Processing image 2
Processing image 3
Processing image 4
Processing image 5
Processing image 6
Processing image 7
Processing image 8
Processing image 9
Processing image 10


Processing image 1
Processing image 2
Processing image 3
Processing image 4
Processing image 5
Processing image 6
Processing image 7
Processing image 8
Processing image 9
Processing image 10


Processing image 1
Processing image 2
Processing image 3
Processing image 4
Processing image 5
Processing image 6
Processing image 7
Processing image 8
Processing image 9
Processing image 10


EME for median filtered images

Processing image 1
Processing image 2
Processing image 3
Processing image 4
Processing image 5
Processing image 6
Processing image 7
Processing image 8
Processing image 9
Processing image 10


Processing image 1
Processing image 2
Processing image 3
Processing image 4
Processing image 5
Processing image 6
Processing image 7
Processing image 8
Processin

### Plotting the metrics for each image (10 plots total)

In [ ]:
import matplotlib.pyplot as plt
if not os.path.exists("plots"):
    os.makedirs("plots")
    os.makedirs("plots/EME")
    os.makedirs("plots/EMEE")
    os.makedirs("plots/AME")
    print(f"Created folder: plots")
else:
    print(f"Folder plots already exists.")

for i in range (0,10):
    print("Creating plots for image " + str(i+1))
    eme = part_a_results[i]["all_values"]
    emee = part_c_results[i]["all_values"]
    ame = part_e_results[i]["all_values"]


# Extract values
    x_axis = list(range(256))
    y_eme  = [eme[i] for i in x_axis]
    y_emee = [emee[i] for i in x_axis]
    y_ame  = [ame[i] for i in x_axis]

    # 3. Create a new empty plot
    #EME
    plt.figure(figsize=(10, 5)) 
    plt.plot(x_axis, y_eme, label='EME', color='blue')
    plt.axvline(x=part_a_results[i]["t_optimal"], color='r', linestyle='--', label=f"t_opt={part_a_results[i]['t_optimal']}")
    plt.scatter(part_a_results[i]["t_optimal"], part_a_results[i]["max_eme"], color='r', zorder=5)
    # X-axis number labels (without this the last tick is 250)
    plt.xticks([0, 50, 100, 150, 200, 255])
    plt.title(f'Filtered Image {i+1} - EME')
    plt.legend()
    plt.savefig(f"plots/EME/filtered_image_{i+1}_eme.png", dpi=300, bbox_inches='tight')
    
    plt.close()

    # EMEE
    plt.figure(figsize=(10, 5)) # Starts a second fresh blank plot
    plt.plot(x_axis, y_emee, label='EMEE', color='red')
    plt.axvline(x=part_c_results[i]["t_optimal"], color='r', linestyle='--', label=f't_opt={part_c_results[i]["t_optimal"]}')
    plt.scatter(part_c_results[i]["t_optimal"], part_c_results[i]["max_eme"], color='r', zorder=5)
    plt.xticks([0, 50, 100, 150, 200, 255])
    plt.title(f'Filtered Image {i+1} - EMEE')
    plt.legend()
    plt.savefig(f"plots/EMEE/filtered_image_{i+1}_emee.png", dpi=300, bbox_inches='tight') 
    
    plt.close()

    #AME
    plt.figure(figsize=(10, 5)) # Starts a third fresh blank plot
    plt.plot(x_axis, y_ame, label='AME', color='green')
    plt.axvline(x=part_e_results[i]["t_optimal"], color='r', linestyle='--', label=f't_opt={part_e_results[i]["t_optimal"]}')
    plt.scatter(part_e_results[i]["t_optimal"], part_e_results[i]["max_eme"], color='r', zorder=5)
    plt.xticks([0, 50, 100, 150, 200, 255])
    plt.title(f'Filtered Image {i+1} - AME')
    plt.legend()   
    plt.savefig(f"plots/AME/filtered_image_{i+1}_ame.png", dpi=300, bbox_inches='tight')
    plt.close()

print("\nDone plotting!")

Created folder: plots
Creating plots for image 1
Creating plots for image 2
Creating plots for image 3
Creating plots for image 4
Creating plots for image 5
Creating plots for image 6
Creating plots for image 7
Creating plots for image 8
Creating plots for image 9
Creating plots for image 10

Done plotting!
